# Dataframe cleanups

En este laboratorio vamos a ver un poco como "limpiar" tablas

### SETUP

In [1]:
!pip install -q bs4
!pip install -q requests pandas

Importamos librerias

In [2]:
import pandas as pd
import requests  
from io import StringIO

## Scraping

Para el scraping primero definimos la URL de la página

In [3]:
URL = "https://en.wikipedia.org/wiki/List_of_countries_by_carbon_dioxide_emissions_per_capita"

Una vez que tenemos la URL tenemos que hacer una solicitud GET a la página y obtener el documento contenido en la respuesta

In [4]:
# Definimos un header que imita un buscador para bypassear el chequeo de scrapping (perdón Wikipedia, solo voy a correr el código una vez)
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36"
}

# Mandamos request GET para obtener el documento
response = requests.get(URL, headers=headers)


En general es importante tener un mínimo entendimiento de como funcionan las páginas web (fundamentalmente conocer la sintáxis HTML) para poder procesar páginas no tan estructuradas como Wikipedia o para procesamientos masivos. <br> Sin embargo, pandas nos da un método para obtener tablas que funciona en Wikipedia, este método nos da una lista de todas las tablas encontradas y transformadas a DataFrames. 

In [5]:
# Transformamos el texto a documento, requerido por metodo read_html
html_asDocument = StringIO(response.text)

# Pandas lee el texto de la respuesta a nuestra request y nos da la lista de tablas
tables = pd.read_html(html_asDocument)

# La tabla que queremos es la primera
df = tables[0] 

# Sampleamos 5 items para verificar que sea la tabla requerida
df.head()


Location % of global average Emissions per capita (tons per year)  \
         Location % of global average                                 2023   
0           World                100%                                 4.86   
1  European Union                117%                                 5.66   
2      Palau[n 4]               1288%                                62.59   
3           Qatar                896%                                43.55   
4          Kuwait                512%                                24.90   

          % change from 2000  
     2000 % change from 2000  
0    4.19               +16%  
1    8.32               −32%  
2  110.66               −43%  
3   53.61               −19%  
4   26.56                −6%

## Limpieza del dataframe

En general la información que obtenemos no puede trabajarse *"as is"* y necesitamos limpiarlo para obtener los formatos requeridos, obviamente esta limpieza es arbitraria y dependerá de nuestros objetivos. 

Por lo pronto analicemos la estructura del dataframe

In [6]:
df.dtypes

Location                              Location                object
% of global average                   % of global average     object
Emissions per capita (tons per year)  2023                   float64
                                      2000                   float64
% change from 2000                    % change from 2000      object
dtype: object

Notemos que la tabla tiene 2 títulos por columna, idealmente querríamos tener un solo título y por lo tanto podemos combinarlos para mayor claridad

In [7]:
df.columns = df.columns.droplevel(0)

df.head()

,Location,% of global average,2023,2000,% change from 2000
0,World,100%,4.86,4.19,+16%
1,European Union,117%,5.66,8.32,−32%
2,Palau[n 4],1288%,62.59,110.66,−43%
3,Qatar,896%,43.55,53.61,−19%
4,Kuwait,512%,24.90,26.56,−6%


Esto nos simplifica un poco la visualización pero simultaneamente perdimos declaratividad, por suerte esto es corregible

In [8]:
retitled_df = df.rename(columns={"2023": "Emissions p.c. 2023", "2000": "Emissions p.c. 2000"})

retitled_df.head()

,Location,% of global average,Emissions p.c. 2023,Emissions p.c. 2000,% change from 2000
0,World,100%,4.86,4.19,+16%
1,European Union,117%,5.66,8.32,−32%
2,Palau[n 4],1288%,62.59,110.66,−43%
3,Qatar,896%,43.55,53.61,−19%
4,Kuwait,512%,24.90,26.56,−6%


Ahora sí podemos continuar el preprocesamiento con un dataframe más atómico

Volviendo a los tipos de nuestras columnas, sería prudente tener la información porcentual en valores numéricos para facilitar su aplicación, si bien generalmente es una transoformación sencilla en casos de string a numero tenemos que asegurarnos de no tener caracteres que rompan el esquema numérico, transformemos las columnas `% of global average` y `% of change from 2000` haciendo lo siguiente:

-   Caso `% of global average`
    1.  Eliminamos el caractér "%" de las columnas 
    2.  Reemplazamos el típo de de la columna de String a Float

-   Caso `% change from 2000`
    1.  Eliminamos el caractér "%" de las columnas
    2.  Reemplazamos el caractér "−" por "-" (notese que son 2 distintos) para retener el signo del número. 
    3.  Reemplaamos el tipo de la columna de String a Float

In [9]:
#Eliminamos de las columnas cualquier caracter no numérico
retitled_df["% of global average"] = retitled_df["% of global average"].str.replace("%", "", regex=False)
retitled_df["% change from 2000"] = retitled_df["% change from 2000"].str.replace("%", "", regex=False).str.replace("−", "-", regex=False).str.replace(",", ".", regex=False)

#Modificamos el tipo de String a Float
retitled_df[["% change from 2000", "% of global average"]] = retitled_df[["% change from 2000", "% of global average"]].astype(float)

Por último vemos que la columna `Location` retuvo patrones de texto HTMl para notas. _(Por ejemplo `Palau[n 4]`)_

> Usando patrones Regex podemos eliminar cualquier substring contenido en dos corchetes con el patrón `\[.*?\]`.

In [10]:
retitled_df["Location"] = retitled_df["Location"].str.replace(r"\[.*?\]", "", regex=True)

retitled_df.head()

,Location,% of global average,Emissions p.c. 2023,Emissions p.c. 2000,% change from 2000
0,World,100.0,4.86,4.19,16.0
1,European Union,117.0,5.66,8.32,-32.0
2,Palau,1288.0,62.59,110.66,-43.0
3,Qatar,896.0,43.55,53.61,-19.0
4,Kuwait,512.0,24.90,26.56,-6.0


Listo! Nuestro dataframe quedó en un formato legible y fácilmente trabajable para gráficos, estadísticas y modelos de ML.

_Se podría limpiar aún más eliminando registros con campos vacíos, pero estaríamos borrando información y eso es responsabilidad del equipo que trabaje los datos_

In [11]:
retitled_df.dtypes

Location                object
% of global average    float64
Emissions p.c. 2023    float64
Emissions p.c. 2000    float64
% change from 2000     float64
dtype: object